<a href="https://colab.research.google.com/github/aditi-govindu/gemini-recipe-agent/blob/main/recipe_agent_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# About
This colab is an introduction to a RAG based ReAct agent that looks at a recipe book, reasons the best course of action and generates a response.

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to aistudio.google.com
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0



In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [3]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [4]:
# @title Create a database of recipes.
manga = {
    "one_piece": (
        "One Piece"
        "Eiichiro Oda"
        "1997"
        "Adventure, Action, Fantasy"
        "Monkey D. Luffy and his crew sail across the Grand Line in search of the legendary One Piece treasure."
    ),

    "aot": (
        "Attack on Titan"
        "Hajime Isayama"
        "2009"
        "Action, Dark Fantasy, Drama"
        "Humanity fights for survival against terrifying Titans while uncovering the secrets behind their existence."
    ),

    "parasite": (
        "Parasyte"
        "Hitoshi Iwaaki"
        "1989"
        "Horror, Sci-Fi, Psychological"
        "A high school student named Shinichi struggles to survive after an alien parasite takes control of his right hand."
    )
}

In [5]:
# @title Define a tool
def search_manga(query: str) -> str:
    """Search the cookbook collection for recipes relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in manga.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant recipes found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Quick test
print(search_manga("one piece"))


[Source: one_piece]
One PieceEiichiro Oda1997Adventure, Action, FantasyMonkey D. Luffy and his crew sail across the Grand Line in search of the legendary One Piece treasure.


In [6]:
# @title Ask gemini to search for something
from google import genai
from google.genai import types
import time
import os

# 1. Initialize the client (it automatically picks up GOOGLE_API_KEY from os.environ)
client = genai.Client()

# 2. Define your model name as a string
MODEL_NAME = "gemini-2.5-flash"  # or "gemini-2.5-pro"

# 3. Define the config including your system instructions and tools
config = types.GenerateContentConfig(
    tools=[search_manga],
    system_instruction=(
        "You are a manga expert helping users explore manga series. "
        "When you need specific information about a manga, such as the author, "
        "publication year, genre, or plot details, use the search_manga tool. "
        "Only answer based on the information you find in the documents. "
        "If the documents do not contain the requested information, clearly say "
        "that the information is not available in the documents."
    )
)

print("Model ready:", MODEL_NAME)

Model ready: gemini-2.5-flash


In [15]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

# Updated model name
MODEL_NAME = "gemini-2.5-flash"  # or "gemini-3.8-flash" depending on quota availability

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_manga",
        description="Search the mangabook to find most the relevant manga to read.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=contents,
                config=config
            )
        except Exception as e:
            # Catch both rate limits (429) and temporary service outages (503)
            if ("429" in str(e) or "503" in str(e)) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Server busy/rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_manga(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"




In [14]:
questions = [
    "Who is jackie chan",
    "Who is the author of Attack on Titan?",
    "What genre is Parasyte?",
    # This question is not answered by our database.
    "How many copies of One Piece have been sold?",
    # This question is part of the general knowledge of the model.
    "What is the most popular manga in Japan?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    try:
        run_agent(q)
    except Exception as e:
        print(f"Error processing question '{q}': {e}")



Question: Who is jackie chan
--------------------------------------------------
Error processing question 'Who is jackie chan': 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}

Question: Who is the author of Attack on Titan?
--------------------------------------------------
Error processing question 'Who is the author of Attack on Titan?': 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}

Question: What genre is P

# Next steps:
* Save a copy of this code to your GitHub repo.
* Create a tool to recommend ingredient substitutions for recipes.
* Push the new tool to your colab and v2 to the same GitHub repo.